# Sub-project 4: Demo Gradio

Tải ảnh món ăn lên, có thể nhập thêm text, rồi xem top-5 của 3 nhánh: chỉ ảnh, chỉ text, fusion (`xattn_strict`). Có thể bật trích xuất JSON bằng Qwen3-VL.

**Cần có trước:** các run `image`, `text_strict`, `xattn_strict` của Mốc 2 trên Drive.

**Bảo mật:** `share=True` tạo link công khai. Thêm Colab Secrets `DEMO_USERNAME` và `DEMO_PASSWORD` trước khi chạy. Nếu thiếu, script sẽ từ chối tạo link.

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/viethoang2503/food-recognize-extract.git"
REPO_DIR = Path("/content/food-recognize-extract")
WORK_DIR = Path("/content/drive/MyDrive/foodmm")
DATA_ROOT = Path("/content/data/upmc_food101")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
WORK_DIR.mkdir(parents=True, exist_ok=True)
BASE = [f"paths.data_root={DATA_ROOT}", f"paths.work_dir={WORK_DIR}"]
print("repo:", REPO_DIR, "| work dir:", WORK_DIR)

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
def run(script, *args):
    """Run a repo script and stream its output into the notebook."""
    cmd = [sys.executable, "-u", f"scripts/{script}", *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{script} failed with exit code {proc.returncode}")

## Dữ liệu
Giải nén bản zip trên Drive về `/content` (lần đầu tải từ Kaggle, cần Colab Secrets `KAGGLE_USERNAME`, `KAGGLE_KEY`). `prepare_data.py` bỏ qua nếu manifest đã có từ Mốc 1.

In [ ]:
import zipfile
from google.colab import userdata

ZIP_PATH = WORK_DIR / "data" / "upmcfood101.zip"
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
if not ZIP_PATH.exists():
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    subprocess.run(["kaggle", "datasets", "download", "-d", "gianmarco96/upmcfood101", "-p", str(ZIP_PATH.parent)], check=True)
if not DATA_ROOT.exists():
    DATA_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)
run("prepare_data.py", "--set", *BASE)

## 1. Tạo ví dụ có sẵn
Chạy một lần. Thêm `--with_vlm` nếu muốn lưu cả kết quả VLM (cần GPU, lâu hơn). Tab "Ví dụ có sẵn" vẫn xem được khi không có GPU.

In [ ]:
if not (WORK_DIR / "demo" / "examples.json").exists():
    run("build_demo_examples.py", "--n", "8", "--set", *BASE)

## 2. Chạy demo
Mở link `*.gradio.live` in ra bên dưới và đăng nhập bằng tài khoản trong Colab Secrets. Dừng cell để tắt demo.

In [ ]:
from google.colab import userdata

os.environ["DEMO_USERNAME"] = userdata.get("DEMO_USERNAME")
os.environ["DEMO_PASSWORD"] = userdata.get("DEMO_PASSWORD")
run("demo.py", "--share", "--set", *BASE)